Survey Aggregated Data Cleaning
===============================

Aim is to clean and preprocess the survey aggregated data to reach `ad breaks level per row` dataset.

# 0/ Import libraries

In [ ]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 1/ Load data

In [ ]:
path= "../../data/raw/experiment_and_survey/< RemovedDueToNDA >_survey_aggregated.xlsx"
df = pd.read_excel(path, sheet_name="table_survey-answers_post-no-ig", header=None) 

# Creating proper column names
df.columns = df.iloc[0:3].fillna("").agg('/'.join, axis=0).str.strip('/').str.replace('-', '_')
df.drop(index=[0,1,2], inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
print("initial df shape:", df.shape)
df.sort_values('< RemovedDueToNDA >_calculated').head()

In [ ]:
df.info()

## Filter df out based on thesis criteria

Keeping only 
- 'lineare' experiment type, 
- exclude participants with exclude_flag = True, and 
- only include participants who answered 'si' to question regarding ads break.

In [ ]:
data = df.copy()
data = data[
            (data['exclude_flag'] != 'True') &              # exclude participants with exclude_flag = True
            (data['experiment_type'] == 'lineare') &        # only include 'lineare' experiment type
            (data['Q3'] == 'Sì') &                          # only include participants who answered 'Sì' to question regarding ads break
            (data['experiment_sequence'] != 3)              # exclude experiment sequence 3 (only 1 participant)   
]

data.shape

## Creation of separated tables

While the previous categorization is valid, a more structured and relational approach would be better for data analysis. This method organizes the data into separate tables based on primary entities, minimizing redundancy and improving data integrity.

In [ ]:
# separation of tables

# === Common Identifier Columns ===
id_cols = [
    'path',
    # 'file_name',                             #redundant with path
    # '< RemovedDueToNDA >',                        #redundant with < RemovedDueToNDA >_calculated
    '< RemovedDueToNDA >_calculated',  
    '< RemovedDueToNDA >',
]

# === Participants Table ===
participant_cols = [
    'birth_year_hard_coded',
    'age',
    'gender_hard_coded',
]

# === Experiments Table ===
experiment_cols = [
    'exclude_flag',                         
    'exclude_reason',
    'pc_recording',
    'study_imotions',
    'experiment_type',
    'device',
    'lineare_duration_pod_s',
    'non_lineare_platform',
    'experiment_sequence',
    'date',
    'time',
    'datetime',
    'content_lineare',
    'platform_on_demand',
    'content_on_demand',
    'content_youtube',
    'adv_pre_roll_1_of_2',
    'adv_pre_roll_2_of_2',
    'adv_mid_roll_1_of_2',
    'adv_mid_roll_2_of_2',

    # Brands -----------
    '< RemovedDueToNDA >',

    # Categories -----------
    'has_auto_ads',
    'has_bev_ads',
    'has_cosm_ads',
    'has_integr_ads',
    'has_arred_ads',
    'has_alim_ads',
    'has_tel_ads',
    'has_ecomm_ads',
    'has_super_ads',
    
    # number of ads per category ----------
    'n_bev_ads',
    'n_alim_ads',
]

# === Survey Responses Table ===
survey_cols = [
    'Q1_1',
    'Q1_2',
    'Q1_3',
    'Q1_4',
    'Q1_5',
    'Q3',
    'Automobili/auto/Q5_auto_dummy',
    'Bevande/bev/Q5_bev_dummy',
    'Cosmetica e profumeria/cosm/Q5_cosm_dummy',
    'Integratori e farmaci/integr/Q5_integr_dummy',
    'Negozi di arredamento e accessori casa/arred/Q5_arred_dummy',
    'Prodotti alimentari/alim/Q5_alim_dummy',
    'Servizi telefonici/internet/tel/Q5_tel_dummy',
    'Siti di e_commerce/ecomm/Q5_ecomm_dummy',
    'Supermercati/super/Q5_super_dummy',
    'Non ricordo//Q5_nonricordo_dummy',
    'Altro (specificare)//Q5_altro',
    'Biscotti/merendine/biscotti/Alim1_biscotti_dummy',
    'Pasta/pasta/Alim1_pasta_dummy',
    'Prodotti da frigo/frigo/Alim1_frigo_dummy',
    'Sughi e conserve/sughi/Alim1_sughi_dummy',
    'Surgelati/surgelati/Alim1_surgelati_dummy',
    'Nessuna di queste/non ricordo//Alim1_nonricordo_dummy',
    'Altro (specificare)//Alim1_6_TEXT',
    'Alim2',
    'Alim3',
    'Alim4',
    'Acqua minerale/acqua/Bev1_acqua_dummy',
    'Bevande vegetali/veg/Bev1_veg_dummy',
    'Bibite gassate/gas/Bev1_gas_dummy',
    'Energy drink/energy/Bev1_energy_dummy',
    'Vino/vino/Bev1_vino_dummy',
    'Nessuna delle precedenti//Bev1_nessuna_dummy',
    'Altro (specificare)//Bev1_7_TEXT',
    'Bev2',
    'Bev3',
    'Bev4',
    'Integr1',
    'Integr2',
    'Integr3',
    'Auto1',
    'Auto2',
    'Auto3',
    'Super1',
    'Super2',
    'Super3',
    'Tel1',
    'Tel2',
    'Tel3',
    'Arred1',
    'Arred2',
    'Arred3',
    'eComm1',
    'eComm2',
    'eComm3',
    'Cosm1',
    'Cosm2',
    'Cosm3',
    'Valutaz1_1',
    'Valutaz1_2',
    'Valutaz1_3',
    'Valutaz1_4',
    'Valutaz1_5'
]

# === Target Variables Table ===
target_cols = [
    # Categories
    'auto/aided_cat_recall_auto_dummy',
    'bev/aided_cat_recall_bev_dummy',
    'cosm/aided_cat_recall_cosm_dummy',
    'integr/aided_cat_recall_integr_dummy',
    'arred/aided_cat_recall_arred_dummy',
    'alim/aided_cat_recall_alim_dummy',
    'tel/aided_cat_recall_tel_dummy',
    'ecomm/aided_cat_recall_ecomm_dummy',
    'super/aided_cat_recall_super_dummy',
    
    # Sub-categories
    'biscotti/aided_sub_cat_recall_biscotti_dummy',
    'pasta/aided_sub_cat_recall_pasta_dummy',
    'sughi/aided_sub_cat_recall_sughi_dummy',
    'acqua/aided_sub_cat_recall_acqua_dummy',
    'veg/aided_sub_cat_recall_veg_dummy',
    'gas/aided_sub_cat_recall_gas_dummy',
    
    # Brands / unaided
< RemovedDueToNDA >
    
    # Brands / aided
< RemovedDueToNDA >
]


In [ ]:
participants = data[id_cols + participant_cols].copy()
print("participants shape:", participants.shape)

experiments = data[id_cols + experiment_cols].copy()
print("experiments shape:", experiments.shape)

surveys = data[id_cols + survey_cols].copy()
print("surveys shape:", surveys.shape)

targets = data[id_cols + target_cols].copy()
print("targets shape:", targets.shape)

# 2/ Data Preprocessing

## Brand Category Mapping

In [ ]:
brand_category_mapping = pd.DataFrame([
    # Alim (Food) category
    {'category': 'alim', 'subcategory': 'biscotti', 'brand': '< RemovedDueToNDA >'},
    {'category': 'alim', 'subcategory': 'pasta', 'brand': '< RemovedDueToNDA >'},
    {'category': 'alim', 'subcategory': 'sughi', 'brand': '< RemovedDueToNDA >'},
    
    # Bev (Beverages) category
    {'category': 'bev', 'subcategory': 'acqua', 'brand': '< RemovedDueToNDA >'},
    {'category': 'bev', 'subcategory': 'veg', 'brand': '< RemovedDueToNDA >'},
    {'category': 'bev', 'subcategory': 'gas', 'brand': '< RemovedDueToNDA >'},
    
    # Auto category
    {'category': 'auto', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Integr (Supplements) category
    {'category': 'integr', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Super (Supermarkets) category
    {'category': 'super', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Tel (Telecom) category
    {'category': 'tel', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Arred (Furniture) category
    {'category': 'arred', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Ecomm (E-commerce) category
    {'category': 'ecomm', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
    
    # Cosm (Cosmetics) category
    {'category': 'cosm', 'subcategory': None, 'brand': '< RemovedDueToNDA >'},
])

brand_category_mapping

## Participant Table

In [ ]:
participants.head()

In [ ]:
participants.info()

In [ ]:
# Fixing data types
participants['birth_year_hard_coded'] = participants['birth_year_hard_coded'].astype('int')
participants['age'] = participants['age'].astype('int')

# Renaming columns for better readability
participants.rename(
    columns={
        'birth_year_hard_coded': 'birth_year',
        'gender_hard_coded' : 'gender'
    },
    inplace=True
)

## Survey Table

In [ ]:
surveys.head()

In [ ]:
surveys.info()

In [ ]:
# fixing data types
temp_num = [ 'Q1_1', 'Q1_2', 'Q1_3', 'Q1_4', 'Q1_5',
            'Valutaz1_1', 'Valutaz1_2', 'Valutaz1_3', 'Valutaz1_4', 'Valutaz1_5']

for col in temp_num:
    surveys[col] = surveys[col].astype('Int64')

# dropping unnecessary columns
surveys.drop(
    columns=[
        'Q3',  # already used for filtering
        
        'Automobili/auto/Q5_auto_dummy',
        'Bevande/bev/Q5_bev_dummy',
        'Cosmetica e profumeria/cosm/Q5_cosm_dummy',
        'Integratori e farmaci/integr/Q5_integr_dummy',
        'Negozi di arredamento e accessori casa/arred/Q5_arred_dummy',
        'Prodotti alimentari/alim/Q5_alim_dummy',
        'Servizi telefonici/internet/tel/Q5_tel_dummy',
        'Siti di e_commerce/ecomm/Q5_ecomm_dummy',
        'Supermercati/super/Q5_super_dummy',
        'Non ricordo//Q5_nonricordo_dummy',
        'Altro (specificare)//Q5_altro',
        'Biscotti/merendine/biscotti/Alim1_biscotti_dummy',
        'Pasta/pasta/Alim1_pasta_dummy',
        'Prodotti da frigo/frigo/Alim1_frigo_dummy',
        'Sughi e conserve/sughi/Alim1_sughi_dummy',
        'Surgelati/surgelati/Alim1_surgelati_dummy',
        'Nessuna di queste/non ricordo//Alim1_nonricordo_dummy',
        'Altro (specificare)//Alim1_6_TEXT',
        'Alim2',
        # 'Alim3',
        'Alim4',
        'Acqua minerale/acqua/Bev1_acqua_dummy',
        'Bevande vegetali/veg/Bev1_veg_dummy',
        'Bibite gassate/gas/Bev1_gas_dummy',
        'Energy drink/energy/Bev1_energy_dummy',
        'Vino/vino/Bev1_vino_dummy',
        'Nessuna delle precedenti//Bev1_nessuna_dummy',
        'Altro (specificare)//Bev1_7_TEXT',
        'Bev2',
        # 'Bev3',
        'Bev4',
        'Integr1',
        #'Integr2',
        'Integr3',
        'Auto1',
        #'Auto2',
        'Auto3',
        'Super1',
        #'Super2',
        'Super3',
        'Tel1',
        #'Tel2',
        'Tel3',
        'Arred1',
        #'Arred2',
        'Arred3',
        'eComm1',
        #'eComm2',
        'eComm3',
        'Cosm1',
        #'Cosm2',
        'Cosm3',
    ], 
    inplace=True)

# Renaming columns for better readability
surveys.rename(
    columns={
        'Q1_1': 'q_content_pleasantness',
        'Q1_2': 'q_content_engagement',
        'Q1_3': 'q_content_interest',
        'Q1_4': 'q_content_quality',
        'Q1_5': 'q_content_production_quality',

        'Valutaz1_1': 'q_ad_quality',
        'Valutaz1_2': 'q_ad_balance',
        'Valutaz1_3': 'q_ad_creativity',
        'Valutaz1_4': 'q_ad_engagement',
        'Valutaz1_5': 'q_ad_brand_influence',
        
        'Alim3': 'seen_ad_before_alim',
        'Bev3': 'seen_ad_before_bev',
        'Integr2': 'seen_ad_before_integr',
        'Auto2': 'seen_ad_before_auto',
        'Super2': 'seen_ad_before_super',
        'Tel2': 'seen_ad_before_tel',
        'Arred2': 'seen_ad_before_arred',
        'eComm2': 'seen_ad_before_ecomm',
        'Cosm2': 'seen_ad_before_cos'
    },
    inplace=True
    )

# reorder columns
surveys = surveys[
    id_cols + 
    [
        'q_content_pleasantness',
        'q_content_engagement',
        'q_content_interest',
        'q_content_quality',
        'q_content_production_quality',
        
        'q_ad_quality',
        'q_ad_balance',
        'q_ad_creativity',
        'q_ad_engagement',
        'q_ad_brand_influence',

        'seen_ad_before_alim',
        'seen_ad_before_bev',
        'seen_ad_before_integr',
        'seen_ad_before_auto',
        'seen_ad_before_super',
        'seen_ad_before_tel',
        'seen_ad_before_arred',
        'seen_ad_before_ecomm',
        'seen_ad_before_cos',
    ]
]

# mapping yes/no to 1/0
seen_ad_before_cols = [ col for col in surveys.columns if 'seen_ad_before' in col ]
for col in seen_ad_before_cols:
    surveys[col] = surveys[col].map({'no': 0, 'No': 0, 'sì': 1, 'Sì': 1, 'nan': pd.NA})

In [ ]:
surveys.describe(percentiles=[0.5], include='all').round(2).T

### Filling missing values in survey responses 

**Strategy Chosen:** `Per-participant imputation with population fallback`

We chose the per-participant imputation strategy because the model aims to predict individual recall, not just population-level averages. Each participant has a unique response pattern (e.g., generally high or low ratings).

For participants who did not answer any questions, we use the overall population median as a fallback. This ensures that missing data is filled without introducing bias from other individuals. Filling missing values with the overall median for participants who answered some questions would erase their individual differences, but using per-participant medians wherever possible preserves personal tendencies and keeps features meaningful for the model.

In [ ]:
ad_score_cols = [col for col in surveys.columns if col.startswith('q_ad_')]
content_score_cols = [col for col in surveys.columns if col.startswith('q_content_')]

rand_participant = surveys['< RemovedDueToNDA >_calculated'].sample(30 , 
                                                               #random_state=123
                                                               )

tempdf = pd.concat([surveys.loc[rand_participant.index].melt(
                        id_vars=['< RemovedDueToNDA >_calculated'],
                        value_vars=ad_score_cols,
                        var_name='ad_question',
                        value_name='ad_score'
                    ),
                    surveys.loc[rand_participant.index].melt(
                        id_vars=['< RemovedDueToNDA >_calculated'],
                        value_vars=content_score_cols,
                        var_name='content_question',
                        value_name='content_score'
                    )],
                    axis=0,
                    ignore_index=True
)

plt.figure(figsize=(8, 3))
sns.kdeplot(data=tempdf, x='ad_score', hue='< RemovedDueToNDA >_calculated', legend=False, alpha=0.3, warn_singular=False)
plt.xlim(0, 10)
plt.title('KDE of Ad Scores by Participant for a sample of participants')
plt.show()

plt.figure(figsize=(8, 3))
sns.kdeplot(data=tempdf, x='content_score', hue='< RemovedDueToNDA >_calculated', legend=False, alpha=0.3, warn_singular=False)
plt.xlim(0, 10)
plt.title('KDE of Content Scores by Participant for a sample of participants')
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(16, 6))
fig.suptitle('Distribution of Survey Scores', fontsize=16, fontweight='bold', y=1.02)

all_score_cols = ad_score_cols + content_score_cols

for i, col in enumerate(all_score_cols):
    r, c = i // 5, i % 5  # Fixed: should be dividing by 5, not 2
    
    # Plot histogram
    ax[r, c].hist(surveys[col].dropna(), bins=10, density=True, alpha=0.4, linewidth=1.2)
    
    # Add mean and median lines
    mean_val = surveys[col].mean()
    median_val = surveys[col].median()
    ax[r, c].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax[r, c].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    
    # Formatting
    ax[r, c].set_xlim(0, 10)
    ax[r, c].set_title(f'{col}\n(n={surveys[col].count()}, NaN={surveys[col].isna().sum()})', fontsize=10, pad=10)
    ax[r, c].set_ylabel('Density', fontsize=9)
    ax[r, c].set_xlabel('Score', fontsize=9)
    ax[r, c].grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
    ax[r, c].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

So we use median as skewness of the columns are high.

In [ ]:
# ==== 1/ per-participant median filling ====
warnings.filterwarnings("ignore")

surveys[ad_score_cols] = surveys.groupby('< RemovedDueToNDA >_calculated')[ad_score_cols].transform(
    lambda g: g.fillna(g.median())
)

surveys[content_score_cols] = surveys.groupby('< RemovedDueToNDA >_calculated')[content_score_cols].transform(
    lambda g: g.fillna(g.median())
)


# ==== 2/ fallback: filling with population median ====
remaining_na = surveys[ad_score_cols+content_score_cols].isna().any(axis=1)

population_median = surveys[ad_score_cols+content_score_cols].median()
surveys.loc[remaining_na, ad_score_cols+content_score_cols] = surveys.loc[remaining_na, ad_score_cols+content_score_cols].fillna(population_median)

# Check if any NaNs remain
print(surveys[ad_score_cols+content_score_cols].isna().sum())

### Seen Ads Before Table

Melting the Survey Question regarding seen ads before columns to ad breaks level

In [ ]:
seen_ad_before = surveys.melt(
    id_vars=['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >'],
    var_name='category',
    value_vars= seen_ad_before_cols,
    value_name='seen_ad_before'
)

seen_ad_before['category'] = seen_ad_before['category'].str.replace('seen_ad_before_', '')

# dropping seen_ad_before columns 
surveys.drop(columns=seen_ad_before_cols, inplace=True)

print(seen_ad_before['category'].unique())

seen_ad_before.head()

> Will be Merged into Experiment Later

## Experiment Table

In [ ]:
experiments.head().T

In [ ]:
experiments.info()

In [ ]:
# fixing data types
temp_num = [ 
            'lineare_duration_pod_s', 
            'experiment_sequence',
            'n_bev_ads', 
            'n_alim_ads' 
          ]

for col in temp_num:
    experiments[col] = experiments[col].astype('int')

experiments['date'] = pd.to_datetime(experiments['date'], format='%d/%m/%Y', errors='coerce')

# drop unnecessary columns
experiments.drop(
      columns=[
              # Already used for filtering
              'exclude_flag', 
              'exclude_reason',
              
              # unnecessary for analysis 
              'time',
              'datetime',
              'pc_recording',
              'study_imotions',
              'experiment_type',
              
              # unrelavant for 'lineare' experiment type
              'platform_on_demand',
              'non_lineare_platform',
              'content_on_demand',
              'content_youtube',
              'adv_pre_roll_1_of_2',
              'adv_pre_roll_2_of_2',
              'adv_mid_roll_1_of_2',
              'adv_mid_roll_2_of_2',
              ], 
      inplace=True
      )

# Renaming columns for better readability
experiments.rename(
  columns={
    'lineare_duration_pod_s': 'ads_break_duration_s',
    'content_lineare': 'content_watched',

    # Advertised Brands
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',
    '< RemovedDueToNDA >': 'ad_< RemovedDueToNDA >',

    # Advertised Categories
    'has_auto_ads': 'ad_auto_cat',
    'has_bev_ads': 'ad_bev_cat',
    'has_cosm_ads': 'ad_cosm_cat',
    'has_integr_ads': 'ad_integr_cat',
    'has_arred_ads': 'ad_arred_cat',
    'has_alim_ads': 'ad_alim_cat',
    'has_tel_ads': 'ad_tel_cat',
    'has_ecomm_ads': 'ad_ecomm_cat',
    'has_super_ads': 'ad_super_cat'
  },
  inplace=True
)

ads_category_cols = [ col for col in experiments.columns if col.startswith('ad_') and col.endswith('_cat')]
ads_brand_cols = [ col for col in experiments.columns if col.startswith('ad_') and not col.endswith('_cat')]

# mapping true/false to 1/0
for col in ads_category_cols + ads_brand_cols:
    experiments[col] = experiments[col].astype('bool').astype(int)

In [ ]:
experiments.info(verbose=False)

In [ ]:
# Checking unique values in content_watched
experiments['content_watched'].value_counts().sort_index()

### Melting the Experiment table to ads level

In [ ]:
# Define ad columns
ad_cols = [c for c in experiments.columns if c.startswith('ad_') and not c.endswith('_cat')]

# 2. Melt and map brands
experiments_long = (
    experiments
    # melting experiments to ad breaks level
    .melt(
        id_vars=['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >'],
        value_vars=ad_cols,
        var_name='name',
        value_name='has_ad'
    )
    
    # extracting brand name from 'name' column
    .assign(brand_name=lambda df: df['name'].str.replace('ad_', ''))
    
    # merging with brand_category_mapping to get brand info
    .merge(brand_category_mapping, left_on='brand_name', right_on='brand', how='left')
    
    # filtering only advertised brands
    .drop(columns=['name', 'brand_name'])
    
    # keeping only rows where has_ad == 1 & necessary columns
    .loc[lambda df: df['has_ad'] == 1, ['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >', 'category', 'subcategory', 'brand', 'has_ad']]
    
    # merging back experiment-level info
    .merge(
        experiments[['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >', 'device',
                     'ads_break_duration_s', 'experiment_sequence', 'date',
                     'content_watched', 'n_bev_ads', 'n_alim_ads']],
        on=['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >'],
        how='left'
    )
)

experiments_long 

## Target Table

In [ ]:
targets.head().T

In [ ]:
targets.info(verbose=False)

In [ ]:
# fixing data types & mapping True/False to 1/0
for col in targets.columns:
    if col not in id_cols:
        targets[col] = targets[col].astype('bool').astype('Int64')
        
# rename columns for better readability
targets.rename(
    columns={
        # Categories / aided
        'auto/aided_cat_recall_auto_dummy':     'recall_aided_cat_auto',
        'bev/aided_cat_recall_bev_dummy':       'recall_aided_cat_bev',
        'cosm/aided_cat_recall_cosm_dummy':     'recall_aided_cat_cosm',
        'integr/aided_cat_recall_integr_dummy': 'recall_aided_cat_integr',
        'arred/aided_cat_recall_arred_dummy':   'recall_aided_cat_arred',
        'alim/aided_cat_recall_alim_dummy':     'recall_aided_cat_alim',
        'tel/aided_cat_recall_tel_dummy':       'recall_aided_cat_tel',
        'ecomm/aided_cat_recall_ecomm_dummy':   'recall_aided_cat_ecomm',
        'super/aided_cat_recall_super_dummy':   'recall_aided_cat_super',

        # Sub-categories / aided
        'biscotti/aided_sub_cat_recall_biscotti_dummy': 'recall_aided_subcat_biscotti',
        'pasta/aided_sub_cat_recall_pasta_dummy':       'recall_aided_subcat_pasta',
        'sughi/aided_sub_cat_recall_sughi_dummy':       'recall_aided_subcat_sughi',
        'acqua/aided_sub_cat_recall_acqua_dummy':       'recall_aided_subcat_acqua',
        'veg/aided_sub_cat_recall_veg_dummy':           'recall_aided_subcat_veg',
        'gas/aided_sub_cat_recall_gas_dummy':           'recall_aided_subcat_gas',

        # Brands / unaided
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                     'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                             'recall_unaided_brand_< RemovedDueToNDA >',
        'alim/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                      'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                             'recall_unaided_brand_< RemovedDueToNDA >',
        'bev/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                   'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                                 'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                           'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                                 'recall_unaided_brand_< RemovedDueToNDA >',
        'alim/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy': 'recall_unaided_brand_< RemovedDueToNDA >',
        'bev/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                             'recall_unaided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                 'recall_unaided_brand_< RemovedDueToNDA >',
        'alim/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                          'recall_unaided_brand_< RemovedDueToNDA >',
        'bev/< RemovedDueToNDA >/unaided_brand_recall_< RemovedDueToNDA >_dummy':                            'recall_unaided_brand_< RemovedDueToNDA >',

        # Brands / aided
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                       'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                               'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                             'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                               'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                         'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                                   'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                             'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                                   'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':     'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                                   'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                   'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                                 'recall_aided_brand_< RemovedDueToNDA >',
        '< RemovedDueToNDA >/aided_brand_recall_< RemovedDueToNDA >_dummy':                                  'recall_aided_brand_< RemovedDueToNDA >'
    },
    inplace=True
)

In [ ]:
brand_unaided_cols = [col for col in targets.columns if col.startswith('recall_unaided_brand_')]
brand_aided_cols = [col for col in targets.columns if col.startswith('recall_aided_brand_')]
cat_aided_cols = [col for col in targets.columns if col.startswith('recall_aided_cat_')]
subcat_aided_cols = [col for col in targets.columns if col.startswith('recall_aided_subcat_')]

### Melting the target table to ads level

In [ ]:
targets_long = targets.melt(
    id_vars=['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >'],
    var_name='recall_item',
    value_name='has_recalled'
)

targets_long[['recall_type', 'level', 'ad_name']] = targets_long['recall_item'].str.extract(r'recall_(aided|unaided)_(cat|subcat|brand)_(.*)')
targets_long = targets_long.drop(columns='recall_item')

#reorder columns
targets_long = targets_long[
    ['path', '< RemovedDueToNDA >_calculated', '< RemovedDueToNDA >', 'recall_type', 'level', 'ad_name', 'has_recalled']
]   

targets_long.head()

We'll need only `Unaided Recall` for `Brands`, so we keep those.

In [ ]:
final_targets_long = targets_long[
    (targets_long['recall_type'] == 'unaided') &
    (targets_long['level'] == 'brand')
].copy()

final_targets_long.drop(columns=['recall_type', 'level'], inplace=True)
final_targets_long.rename(columns={'ad_name': 'brand', 
                            'has_recalled': 'unaided_brand_recall'}, 
                    inplace=True)

final_targets_long

# 3/ Final Data Export

Final check of the tables and saving them as csv files.

In [ ]:
# brand_category_mapping
# experiments_long
# participants
# seen_ad_before
# surveys
# final_targets_long

In [ ]:
ad_break_level_obt = experiments_long\
                .merge(
                    participants,
                    on= id_cols,
                    how='left'
                ).merge(
                    seen_ad_before,
                    on= id_cols + ['category'],
                    how='left'
                ).merge(
                    surveys,
                    on= id_cols,
                    how='left'
                ).merge(
                    final_targets_long,
                    on= id_cols + ['brand'],
                    how='left'
                )

ad_break_level_obt.head()

In [ ]:
# Exporting cleaned data to CSV files
output_dir = "../../data/processed/ad_break_level_meta"
Path(output_dir).mkdir(parents=True, exist_ok=True)

ad_break_level_obt.to_parquet(os.path.join(output_dir, "ad_break_level_PESR_data.parquet"), index=False)

> **Important Note:** This data doesn't contain temporal features yet. Temporal features will be added in the next steps after using `segment-level data` to extract temporal features and merge them into this dataset.

In [ ]:
# Exporting separated tables to CSV files
output_dir = "../../data/processed/ad_break_level_meta/separated_tables/"
Path(output_dir).mkdir(parents=True, exist_ok=True)

surveys.to_parquet(os.path.join(output_dir, "surveys.parquet"), index=False)
experiments.to_parquet(os.path.join(output_dir, "experiments.parquet"), index=False)
experiments_long.to_parquet(os.path.join(output_dir, "experiments_long.parquet"), index=False)
participants.to_parquet(os.path.join(output_dir, "participants.parquet"), index=False)
final_targets_long.to_parquet(os.path.join(output_dir, "targets_long.parquet"), index=False)

---